# Avoiding reconstruction of full tensor
We use a sparse tensor as input, that can be very big before giving any memory issues.
For the tucker decomposition, one of the steps involves reconstructing the full tensor from the core and factors.
This can be very memory intensive: we need to find an alternative.

In [2]:
from tensorly_custom.tucker_tensor import validate_tucker_rank, tucker_normalize, TuckerTensor
from tensorly_custom.decomposition._tucker import tucker_to_tensor
from tensorly_custom.base import unfold
from tensorly_custom.tenalg import mode_dot
import tensorly_custom as tl
import pytensorlab as ptl
import numpy as np
from typing import List, Tuple
from utils import DATA_DIR, select_gpu, tree_to_device, notify_discord
from tucker_tensor import SparseTupleTensor
from typing import Optional, Union
import cupy as cp
import cupyx.scipy.sparse as cpx_sparse
import time


device = select_gpu()
tl.set_backend("cupy")

# ----- cupy sparse methods -----

def fro_norm_coo(x):
    # x: cupyx.scipy.sparse.coo_matrix
    data = x.data
    return cp.sqrt((cp.abs(data) ** 2).sum())

def unfold_from_vectorized_sparse(
    vec_tensor: cpx_sparse.spmatrix,
    orig_shape,
    mode: int,
    to_dense: bool = False,
):
    """
    Unfold a sparse tensor that is stored as a vectorized CuPy sparse matrix.

    Parameters
    ----------
    vec_tensor : cupyx.scipy.sparse.spmatrix
        Sparse matrix of shape (np.prod(orig_shape), 1) created by
        `torch_sparse_to_cupy` for an N-D tensor.
    orig_shape : tuple[int, ...]
        Original N-D tensor shape, e.g. (I0, I1, I2).
    mode : int
        Mode along which to unfold.
    to_dense : bool, default False
        If True, return a dense cupy.ndarray.
        If False, return a cupy sparse COO matrix.

    Returns
    -------
    unfolded : cupy.ndarray or cupyx.scipy.sparse.coo_matrix
        Mode-`mode` unfolding of shape
        (orig_shape[mode], np.prod(orig_shape) // orig_shape[mode]).
    """
    # Make sure we're in COO format

    cu = vec_tensor.tocoo()

    row_cp = cu.row
    col_cp = cu.col
    data_cp = cu.data

    orig_shape = tuple(orig_shape)
    size = int(np.prod(orig_shape))
    int32_max = np.iinfo(np.int32).max
    block_size = min(size, int32_max)

    # ---- move to host and use int64 for safe arithmetic ----
    row_np = cp.asnumpy(row_cp).astype(np.int64)
    col_np = cp.asnumpy(col_cp).astype(np.int64)

    flat_np = row_np + col_np * np.int64(block_size)

    coords = np.unravel_index(flat_np, orig_shape)

    row_unf_np = coords[mode]

    other_coords = coords[:mode] + coords[mode + 1:]
    other_shape = tuple(s for i, s in enumerate(orig_shape) if i != mode)
    col_unf_np = np.ravel_multi_index(other_coords, other_shape)

    row_unf_cp = cp.asarray(row_unf_np)
    col_unf_cp = cp.asarray(col_unf_np)

    unfolded_shape = (orig_shape[mode], int(np.prod(other_shape)))
    unfolded = cpx_sparse.coo_matrix(
        (data_cp, (row_unf_cp, col_unf_cp)),
        shape=unfolded_shape,
    )

    if to_dense:
        return unfolded.toarray()
    return unfolded


def left_dense_mul_sparse(
    mat: cp.ndarray,
    sp: cpx_sparse.spmatrix
) -> Union[cp.ndarray, cpx_sparse.coo_matrix]:
    """
    Compute mat @ sp, choosing dense or sparse output based on a simple
    memory heuristic.

    mat: cupy ndarray of shape (R, I_mode)
    sp:  cupy sparse matrix of shape (I_mode, K)
    """
    sp = sp.tocoo()
    R, I_mode = mat.shape
    assert I_mode == sp.shape[0], f"mat shape {mat.shape} not compatible with sparse {sp.shape}"

    # Let CuPy handle dense @ sparse; result is cupy.ndarray
    return mat @ sp
def fold_unfolded_sparse_to_vec(
    unfolded: Union[cpx_sparse.spmatrix, cp.ndarray],
    old_shape: Tuple[int, ...],
    mode: int,
    new_dim: int,
) -> Tuple[cpx_sparse.coo_matrix, Tuple[int, ...]]:
    """
    Fold a mode-`mode` unfolded matrix back to a vectorized sparse tensor.

    unfolded:
        - sparse COO or any cupyx.scipy.sparse.spmatrix of shape (new_dim, prod(other_dims)), or
        - dense cupy.ndarray of the same shape.
    old_shape : original N-D shape BEFORE replacing dimension at `mode`
    mode      : mode index that was unfolded
    new_dim   : new size at `mode` (typically rank[mode])

    Returns
    -------
    vec_sparse : COO of shape (prod(new_shape), 1)
    new_shape  : tuple of ints, updated tensor shape
    """

    old_shape = tuple(old_shape)
    N = len(old_shape)

    new_shape = list(old_shape)
    new_shape[mode] = new_dim
    new_shape = tuple(new_shape)

    other_shape = tuple(s for i, s in enumerate(old_shape) if i != mode)

    if cpx_sparse.isspmatrix(unfolded):
        unfolded = unfolded.tocoo()
        row = unfolded.row
        col = unfolded.col
        data = unfolded.data
    else:
        row, col = cp.nonzero(unfolded)
        data = unfolded[row, col]

    coords_other = cp.unravel_index(col, other_shape)

    coords_full = []
    idx_other = 0
    for i in range(N):
        if i == mode:
            coords_full.append(row)
        else:
            coords_full.append(coords_other[idx_other])
            idx_other += 1

    coords_full = tuple(coords_full)

    size = int(np.prod(new_shape))
    int32_max = np.iinfo(np.int32).max
    block_size = min(size, int32_max)

    flat = cp.ravel_multi_index(coords_full, new_shape)

    # --- block encoding of flat indices ---
    row_vec = flat % block_size
    col_vec = flat // block_size

    n_blocks = int((size + block_size - 1) // block_size)
    vec_sparse = cpx_sparse.coo_matrix(
        (data, (row_vec, col_vec)),
        shape=(block_size, n_blocks),
    )
    vec_sparse.sum_duplicates()

    return vec_sparse, new_shape


def sparse_mode_dot_vec(
    vec_tensor: cpx_sparse.spmatrix,
    curr_shape: Tuple[int, ...],
    factor: cp.ndarray,
    mode: int,
    transpose_factor: bool = True,
) -> Tuple[cpx_sparse.coo_matrix, Tuple[int, ...]]:
    """
    Perform a mode-`mode` product on a vectorized sparse tensor (prod(curr_shape), 1),
    using a dense factor matrix, and return the new vectorized sparse tensor.

    vec_tensor: sparse COO (prod(curr_shape), 1)
    curr_shape: current tensor shape
    factor:     dense matrix of shape (I_mode, R_mode) (or R_mode, I_mode if transpose_factor=False)
    mode:       mode index in [0, len(curr_shape))
    transpose_factor: if True, use factor.T (for Tucker-style X ×_n W_n^T)

    Returns
    -------
    new_vec:   sparse COO (prod(new_shape), 1)
    new_shape: updated shape, with dimension at `mode` replaced by R_mode
    """
    curr_shape = tuple(curr_shape)
    I_mode = curr_shape[mode]

    # Factor handling
    if transpose_factor:
        # factor is (I_mode, R_mode) => mat is (R_mode, I_mode)
        assert factor.shape[0] == I_mode, f"factor shape {factor.shape} not compatible with dim {I_mode}"
        mat = tl.transpose(factor)  # (R_mode, I_mode)
    else:
        # factor is already (R_mode, I_mode)
        assert factor.shape[1] == I_mode, f"factor shape {factor.shape} not compatible with dim {I_mode}"
        mat = factor

    R_mode = mat.shape[0]

    # 1) Unfold current sparse tensor along this mode (sparse COO)
    unfolded = unfold_from_vectorized_sparse(
        vec_tensor,
        curr_shape,
        mode,
        to_dense=False,
    )  # shape: (I_mode, prod(other_dims))

    # 2) Left-multiply with dense matrix; currently returns dense cp.ndarray
    #    -> shape: (R_mode, prod(other_dims))
    unfolded_new = left_dense_mul_sparse(mat, unfolded)

    # 3) Fold back into a new vectorized sparse tensor with updated shape
    new_vec, new_shape = fold_unfolded_sparse_to_vec(
        unfolded_new,
        old_shape=curr_shape,
        mode=mode,
        new_dim=R_mode,
    )
    return new_vec, new_shape
def sparse_multi_mode_dot_vec(
    vec_tensor: cpx_sparse.spmatrix,
    orig_shape: Tuple[int, ...],
    factors: List[cp.ndarray],
    modes: Optional[List[int]] = None,
    transpose_factors: bool = True,
) -> cp.ndarray:
    """
    multi_mode_dot for a vectorized sparse tensor (prod(orig_shape), 1),
    applying dense factor matrices along the given modes, **staying sparse**
    until the final (small) result, which is densified.

    vec_tensor: sparse COO (prod(orig_shape), 1)
    orig_shape: original tensor shape
    factors:    list of factor matrices, one per mode index
                factor[n] has shape (I_n, R_n)
    modes:      list of modes to apply; if None, uses range(len(factors))
    transpose_factors: if True, uses factors[n].T (Tucker-style)
    """
    if modes is None:
        modes = list(range(len(factors)))

    current_vec = vec_tensor
    current_shape = tuple(orig_shape)

    # Apply each mode in any order (commutes)
    for mode in modes:
        # print("Applying mode", mode)
        current_vec, current_shape = sparse_mode_dot_vec(
            current_vec,
            current_shape,
            factors[mode],
            mode=mode,
            transpose_factor=transpose_factors,
        )

    # At this point, current_vec is still sparse (prod(core_shape), 1)
    core_shape = current_shape  # typically your (50, 50, 50) or similar
    # should not overflow the cupy 32bit index limit if dimensions stay reasonable
    # Finally densify the small core
    coo = current_vec.tocoo()
    flat = coo.row
    data = coo.data

    # Build dense core
    coords = cp.unravel_index(flat, core_shape)
    core_dense = cp.zeros(core_shape, dtype=data.dtype)
    core_dense[coords] = data

    return core_dense

def print_elapsed_time(start_time, message=""):
    """Prints the elapsed time since start_time."""
    now = time.time()
    # if the message starts with indents (tabs), add the same number to the elapsed time print
    tabs = ""
    for char in message:
        if char == "\t":
            tabs += "\t"
        else:
            break
    elapsed_time = now - start_time
    minutes, seconds = divmod(elapsed_time, 60)
    if message:
        print(message)
    print(f"{tabs}Elapsed time: {int(minutes)} minutes and {seconds} seconds.")
    return now

sparse_tensor = SparseTupleTensor.load_from_disk(
    dataset="fineweb_large",
    method="sc",
    dims=1000,
    map_location="cpu"
)
sparse_tensor.tensor_to_sparse("cupy")
n_iter_max=10
init="random"
tol=10e-5
random_state=42
verbose=False
return_errors="full"
normalize_factors=False
patience: int=3
rank = [100, 100, 100]
shape = tuple(sparse_tensor.shape)
tensor = sparse_tensor.tensor
rank = validate_tucker_rank(shape, rank=rank)
epsilon = 1e-12
modes = list(range(len(rank)))
non_negative = True
# skip_factor = None
# transpose_factors = False


if init == "random":
    rng = tl.check_random_state(random_state)
    core = tl.tensor(
        rng.random_sample([rank[index] for index in range(len(modes))]) + 0.01,
        **tl.context(tensor),
    )
    factors = [
        tl.tensor(
            rng.random_sample((shape[mode], rank[index])), # we changed this to original shape
            **tl.context(tensor),
        )
        for index, mode in enumerate(modes)
    ]
else:
    (core, factors) = init

if non_negative is True:
    factors = [tl.abs(f) for f in factors]
    core = tl.abs(core)
else:
    raise NotImplementedError("Currently only non-negative=True is supported.")


Selected GPU 1 with 451.75 MB used memory.


In [2]:
core_np = tl.to_numpy(core)
factors_np = [tl.to_numpy(f) for f in factors]
tl_tucker = ptl.TuckerTensor(core=core_np,
                            factors=factors_np)
tl_sparse = ptl.SparseTensor.from_sparse_matrix(tensor.get()).reshape((1000, 1000, 1000))
for mode in modes:
    X = unfold_from_vectorized_sparse(tensor, shape, mode)
    ptl_X = ptl.tens2mat(tl_sparse, mode)
    print(np.allclose(cp.asnumpy(ptl_X.toarray()), tl.to_numpy(X.toarray())))

True
True
True


In [3]:
for mode in modes:
    tl_factor = ptl.tmprod(tl_tucker.core, [f for i, f in enumerate(tl_tucker.factors) if i != mode],
                         [i for i in range(tl_tucker.ndim) if i != mode])
    tlc_factor = tucker_to_tensor((core, factors), skip_factor=mode)
    assert np.allclose(tl_factor, tlc_factor)

In [3]:
def ptl_tucker_to_tensor(tucker: ptl.TuckerTensor,
                         skip_factor: Optional[int] = None) -> np.ndarray:
    """Reconstruct full tensor from Tucker representation, optionally skipping one factor."""
    factors = tucker.factors
    if skip_factor is not None:
        factors = [f for i, f in enumerate(factors) if i != skip_factor]
    return ptl.tmprod(tucker.core, factors, list(range(tucker.ndim)) if skip_factor is None else
                     [i for i in range(tucker.ndim) if i != skip_factor])

In [5]:
for mode in modes:
    ptl_factor = ptl_tucker_to_tensor(tl_tucker, skip_factor=mode)
    tlc_factor = tucker_to_tensor((core, factors), skip_factor=mode)
    assert np.allclose(ptl_factor, tlc_factor)
    tlc_factor_unfold = unfold(tlc_factor, mode)
    tlc_factor_unfold_np = tl.to_numpy(tlc_factor_unfold)
    ptl_factor_unfold = ptl.tens2mat(ptl_factor, mode)
    assert np.allclose(tlc_factor_unfold_np, ptl_factor_unfold)

In [6]:
Z = tucker_to_tensor((core, factors), skip_factor=0)
Z_unfold = unfold(Z, 0) # (50, 1000000)
Z_np = tl.to_numpy(Z)
Z_unfold_np = tl.to_numpy(Z_unfold)
Z_unfold.shape

(100, 1000000)

In [7]:
ptl_Z = ptl_tucker_to_tensor(tl_tucker, skip_factor=0)
ptl_Z_unfold = ptl.tens2mat(ptl_Z, 0)
ptl_Z_unfold.shape

(100, 1000000)

In [8]:
print(np.allclose(Z_np, ptl_Z), np.allclose(Z_unfold_np, ptl_Z_unfold))

True True


In [9]:
A = factors[0] # (1000, 50)
ptl_A = factors_np[0]
R = tl.dot(A, Z_unfold) # (1000, 1000000)
ptl_R = ptl_A @ ptl_Z_unfold
assert np.allclose(tl.to_numpy(R), ptl_R)

In [10]:
Z_T = tl.transpose(Z) # (1000000, 50)
ptl_Z_T = tl.transpose(ptl_Z_unfold)

In [11]:
X = unfold_from_vectorized_sparse(tensor, shape, 0)
frac = X / R # (1000, 100000)
tl_sparse = ptl.SparseTensor.from_sparse_matrix(tensor.get()).reshape((1000, 1000, 1000))
ptl_X = ptl.tens2mat(tl_sparse, 0)
ptl_frac = ptl_X / ptl_R

frac_np = tl.to_numpy(frac.toarray())
ptl_frac_np = cp.asnumpy(ptl_frac.toarray())
np.allclose(frac_np, ptl_frac_np)

True

In [12]:
start = time.time()
tl_sparse = ptl.SparseTensor.from_sparse_matrix(tensor.get()).reshape((1000, 1000, 1000))
for mode_n in modes:
    print(f"Calculating factor for mode {mode_n}...")
    X = unfold_from_vectorized_sparse(tensor, shape, mode_n)
    ptl_X = ptl.tens2mat(tl_sparse, mode_n)
    assert np.allclose(cp.asnumpy(ptl_X.toarray()), tl.to_numpy(X.toarray()))

    core_np = tl.to_numpy(core)
    factors_np = [tl.to_numpy(f) for f in factors]
    ptl_tucker = ptl.TuckerTensor(core=core_np,
                                factors=factors_np)
    ptl_Z = ptl_tucker_to_tensor(ptl_tucker, skip_factor=mode_n)
    Z = tucker_to_tensor((core, factors), skip_factor=mode_n)
    assert np.allclose(tl.to_numpy(Z), ptl_Z)

    ptl_Z_unfold = ptl.tens2mat(ptl_Z, mode_n)
    Z = unfold(Z, mode_n) # (50, 1000000)
    assert np.allclose(tl.to_numpy(Z), ptl_Z_unfold)

    A = factors[mode_n] # (1000, 50)
    ptl_A = factors_np[mode_n]

    R = tl.dot(A, Z) # (1000, 1000000)
    ptl_R = ptl_A @ ptl_Z_unfold
    assert np.allclose(tl.to_numpy(R), ptl_R)

    Z_T = tl.transpose(Z) # (1000000, 50)
    ptl_Z_T = tl.transpose(ptl_Z_unfold)
    assert np.allclose(tl.to_numpy(Z_T), ptl_Z_T)

    frac = X / R # (1000, 100000)
    frac_np = tl.to_numpy(frac.toarray())

    # this takes a long time
    ptl_frac = ptl_X / ptl_R
    ptl_frac_np = ptl_frac.toarray()
    assert np.allclose(frac_np, ptl_frac_np)

    numerator = frac @ Z_T # (1000, 50)
    num_np = tl.to_numpy(numerator)
    ptl_numerator = ptl_frac @ ptl_Z_T
    # print("shapes:", num_np.shape, ptl_numerator.shape)
    assert np.allclose(num_np, ptl_numerator)

    # Denominator: E Z^T  ==  row-wise broadcast of sum over columns of Z
    den_row = tl.sum(Z, axis=1)    # (R,)
    den_row_np = tl.to_numpy(den_row)
    ptl_den_row = tl.sum(ptl_Z_unfold, axis=1)
    assert np.allclose(tl.to_numpy(den_row), ptl_den_row)

    denominator = den_row[np.newaxis, :]   # (1, R) – will broadcast to (I, R)
    ptl_denominator = ptl_den_row[np.newaxis, :]
    assert np.allclose(tl.to_numpy(denominator), ptl_denominator)

    # Multiplicative update
    A = A * (numerator / (denominator + 1e-12))
    A_np = tl.to_numpy(A)
    ptl_A = ptl_A * (ptl_numerator / (ptl_denominator + 1e-12))
    assert np.allclose(A_np, ptl_A)

    A = tl.clip(A, a_min=epsilon, a_max=None)
    A_np = tl.to_numpy(A)
    ptl_A = tl.clip(ptl_A, a_min=epsilon, a_max=None)
    assert np.allclose(A_np, ptl_A)

    factors[mode_n] = A

a = print_elapsed_time(start, "KL factors calculated:")

Calculating factor for mode 0...
Calculating factor for mode 1...
Calculating factor for mode 2...


AssertionError: 

In [13]:
dims = 1000
sparse_tensor = SparseTupleTensor.load_from_disk(
    dataset="fineweb_large",
    method="sc",
    dims=dims,
    map_location="cpu"
)
sparse_tensor.tensor_to_sparse("sparse")
ptl_tensor = ptl.SparseTensor(data=sparse_tensor.tensor.data,
                        indices=sparse_tensor.tensor.coords,
                        shape=sparse_tensor.tensor.shape)
sparse_tensor.tensor_to_sparse("torch")
sparse_tensor.tensor_to_sparse("cupy")
rank = [100, 100, 100]
tensor = sparse_tensor.tensor
shape = tuple(sparse_tensor.shape)
modes = list(range(len(rank)))

if init == "random":
    rng = tl.check_random_state(random_state)
    core = tl.tensor(
        rng.random_sample([rank[index] for index in range(len(modes))]) + 0.01,
        **tl.context(tensor),
    )
    factors = [
        tl.tensor(
            rng.random_sample((shape[mode], rank[index])), # we changed this to original shape
            **tl.context(tensor),
        )
        for index, mode in enumerate(modes)
    ]
else:
    (core, factors) = init

if non_negative is True:
    factors = [tl.abs(f) for f in factors]
    core = tl.abs(core)
else:
    raise NotImplementedError("Currently only non-negative=True is supported.")

In [14]:
print("shape:", shape)
print("rank:", rank)
print("sparse_tensor shape:", sparse_tensor.shape)
print("core shape:", core.shape)
print("factors shapes:", [f.shape for f in factors])
print("ptl_tensor shape:", ptl_tensor.shape)

shape: (1000, 1000, 1000)
rank: [100, 100, 100]
sparse_tensor shape: torch.Size([1000, 1000, 1000])
core shape: (100, 100, 100)
factors shapes: [(1000, 100), (1000, 100), (1000, 100)]
ptl_tensor shape: (1000, 1000, 1000)


In [2]:
import numpy as np
import cupy as cp
from cupyx.scipy import sparse
import torch

def sparse_dense_division_batched(X_csr_gpu, R_host, batch_rows=10_000):
    """
    X_csr_gpu : cupyx.scipy.sparse.csr_matrix on GPU
    R_host    : numpy.ndarray on CPU with same shape as X
    batch_rows: number of rows to process per batch
    """
    N, M = X_csr_gpu.shape
    assert R_host.shape == (N, M)

    # Store GPU sparse chunks and vstack at end
    batches = []
    for start in range(0, N, batch_rows):
        stop = min(start + batch_rows, N)

        # Sparse slice is still on GPU (cheap)
        X_batch_gpu = X_csr_gpu[start:stop]

        # Move matching dense slice to GPU
        R_batch_gpu = cp.asarray(R_host[start:stop])

        # Elementwise division only for nonzeros (sparse × dense)
        # multiply is implemented efficiently on sparse
        Y_batch_gpu = X_batch_gpu.multiply(1.0 / R_batch_gpu)

        batches.append(Y_batch_gpu)

    # Combine back into one sparse matrix on GPU
    Y_gpu = sparse.vstack(batches, format='csr')
    return Y_gpu


In [16]:
skip_factor_tensors = [ptl.tens2mat(ptl_tensor, mode) for mode in modes]
max_cuda_mem = torch.cuda.get_device_properties().total_memory
batch_rows = max_cuda_mem // (dims ** 2 * 12)

# Proof of equivalence of the three methods

In [17]:
X = unfold_from_vectorized_sparse(tensor, shape, 0)
Z = tucker_to_tensor((core, factors), skip_factor=0)
Z_unfold = unfold(Z, 0) # (50, 1000000)
R = tl.dot(factors[0], Z_unfold) # (1000, 1000000)
frac = X / R # (1000, 100000)

ptl_X = skip_factor_tensors[0]
ptl_frac = ptl_X / ptl_R

frac_np = tl.to_numpy(frac.toarray())
ptl_frac_np = cp.asnumpy(ptl_frac.toarray())
print(np.allclose(frac_np, ptl_frac_np))

X_csr = sparse.csr_matrix(X)
R_np = tl.to_numpy(R)
frac_batched = sparse_dense_division_batched(X_csr, R_np, batch_rows=batch_rows)
frac_batched_np = cp.asnumpy(frac_batched.toarray())
print(np.allclose(frac_np, frac_batched_np))

False
True


In [22]:
start = time.time()

for mode_n in modes:
    start_mode = time.time()
    # X = unfold_from_vectorized_sparse(tensor, shape, mode_n)
    # X = ptl.tens2mat(ptl_tensor, mode_n)
    X = skip_factor_tensors[mode_n]
    core_np = tl.to_numpy(core)
    factors_np = [tl.to_numpy(f) for f in factors]
    ptl_tucker = ptl.TuckerTensor(core=core_np,
                                factors=factors_np)
    Z = ptl_tucker_to_tensor(ptl_tucker, skip_factor=mode_n)
    # Z = tucker_to_tensor((core, factors), skip_factor=mode_n)
    # Z = unfold(Z, mode_n) # (50, 1000000)
    Z = ptl.tens2mat(Z, mode_n)
    A = factors[mode_n] # (1000, 50)
    A_np = factors_np[mode_n]
    # print("Z:", type(Z), Z.shape, "\nA:", type(A), A.shape)

    # this breaks.
    # R = tl.dot(A, Z) # (1000, 1000000)
    pre_R_time = print_elapsed_time(start_mode, "factors prepared")
    R = A_np @ Z # (1000, 1000000)
    post_R_time = print_elapsed_time(pre_R_time, "R calculated:")
    # print(f"R type: {type(R)}, shape {R.shape}")
    Z_T = tl.transpose(Z) # (1000000, 50)
    # print(f"Z_T type: {type(Z_T)}, shape {Z_T.shape}")
    # print(f"X type: {type(X)}, shape {X.shape}")

    X_csr_gpu = sparse.csr_matrix(X)
    frac = sparse_dense_division_batched(X_csr_gpu, R, batch_rows=batch_rows)
    ptl_frac = X / R
    ptl_frac_np = ptl_frac.toarray()


    post_frac_time = print_elapsed_time(post_R_time, "frac calculated:")
    Z_T_cp = cp.asarray(Z_T)
    convert_time = print_elapsed_time(post_frac_time, "Z_T converted to cupy:")
    numerator = frac @ Z_T_cp # (1000, 50)
    post_numerator_time = print_elapsed_time(convert_time, "numerator calculated:")
    # print(f"numerator type: {type(numerator)}, shape {numerator.shape}")
    den_row = tl.sum(tl.transpose(Z_T_cp), axis=1)    # (R,)
    denominator = den_row[np.newaxis, :]   # (1, R) – will broadcast to (I, R)
    # print(f"denominator type: {type(denominator)}, shape {denominator.shape}")
    # Multiplicative update
    denom_time = print_elapsed_time(post_numerator_time, "denominator calculated:")
    A = A * (numerator / (denominator + 1e-12))
    A = tl.clip(A, a_min=epsilon, a_max=None)

    factors[mode_n] = A
    del numerator, denominator, frac, X_csr_gpu, Z_T_cp, R, Z
    torch.cuda.empty_cache()
    update_time = print_elapsed_time(denom_time, f"Factor for mode {mode_n} updated.")

a = print_elapsed_time(start, "KL factors calculated:")

NameError: name 'skip_factor_tensors' is not defined

In [19]:

mode_n = 0
X = skip_factor_tensors[mode_n]
core_np = tl.to_numpy(core)
factors_np = [tl.to_numpy(f) for f in factors]
ptl_tucker = ptl.TuckerTensor(core=core_np,
                            factors=factors_np)
Z = ptl_tucker_to_tensor(ptl_tucker, skip_factor=mode_n)
# Z = tucker_to_tensor((core, factors), skip_factor=mode_n)
# Z = unfold(Z, mode_n) # (50, 1000000)
Z = ptl.tens2mat(Z, mode_n)
A = factors[mode_n] # (1000, 50)
A_np = factors_np[mode_n]
# print("Z:", type(Z), Z.shape, "\nA:", type(A), A.shape)

# this breaks.
# R = tl.dot(A, Z) # (1000, 1000000)
R = A_np @ Z
X_csr_gpu = sparse.csr_matrix(X)


In [20]:
max_cuda_mem = torch.cuda.get_device_properties().total_memory
batch_rows = max_cuda_mem // (dims ** 2 * 12)
print(batch_rows)
frac = sparse_dense_division_batched(X_csr_gpu, R, batch_rows=batch_rows)

2108


# Hybrid for efficient large-scale tensor computations

In [4]:
from sparse_ops import initialize_nonnegative_tucker
dataset = "fineweb_large"
method = "sc"
dim = 2000
def inspect(element):
    element_var = globals()[element]
    print(f"{element}, {element_var.shape}, {type(element_var)}")
    ptl_el = "ptl_"+element
    try:
        inspect(ptl_el)
    except:
        return

sparse_tensor = SparseTupleTensor.load_from_disk(
                dataset=dataset,
                method=method,
                dims=dim,
                map_location="cpu",
            )
sparse_tensor.tensor_to_sparse("sparse")
ptl_tensor = ptl.SparseTensor(data=sparse_tensor.tensor.data,
                        indices=sparse_tensor.tensor.coords,
                        shape=sparse_tensor.tensor.shape)
sparse_tensor.tensor_to_sparse("torch")
sparse_tensor.tensor_to_sparse("cupy")
sparse_tensor.tensor_to_sparse("cupy")
rank = [150, 150, 150]
tensor = sparse_tensor.tensor
shape = tuple(sparse_tensor.shape)
modes = list(range(len(rank)))
core, factors = initialize_nonnegative_tucker(sparse_tensor, shape, rank, modes, init, random_state)

In [45]:
mode = 0
# Sparse unfolding for this mode
X = unfold_from_vectorized_sparse(tensor, shape, mode)
core_np = tl.to_numpy(core)
factors_np = [tl.to_numpy(f) for f in factors]
# Dense reconstruction excluding current factor, unfolded along mode
ptl_tucker = ptl.TuckerTensor(core=core_np,
                                factors=factors_np)
ptl_Z = ptl_tucker_to_tensor(ptl_tucker, skip_factor=mode)
ptl_Z = ptl.tens2mat(ptl_Z, mode)
Z = tucker_to_tensor((core, factors), skip_factor=mode)
Z = unfold(Z, mode)  # (R, K) after unfold
inspect("Z")

Z, (150, 16000000), <class 'cupy.ndarray'>
ptl_Z, (150, 16000000), <class 'numpy.ndarray'>


In [46]:
A = factors[mode]  # (I_mode, R)
rows = X.row
cols = X.col
vals = X.data

# Compute reconstruction only at nonzeros: R_nz = sum_r A[i,r] * Z[r,j]
# A_rows: (nnz, R)
A_rows = A[rows, :]
# Z_cols_T: (nnz, R) because Z[:, cols] is (R, nnz)
Z_cols_T = tl.transpose(Z[:, cols])
ptl_Z_cols_T = tl.transpose(ptl_Z[:, cols.get()])
ptl_Z_cols_T = cp.asarray(Z_cols_T)
R_nz = tl.sum(A_rows * Z_cols_T, axis=1)
R_nz = tl.clip(R_nz, a_min=epsilon, a_max=None)
ptl_R_nz = tl.sum(A_rows * ptl_Z_cols_T, axis=1)
ptl_R_nz = tl.clip(ptl_R_nz, a_min=epsilon, a_max=None)
inspect("Z_cols_T")
inspect("R_nz")


Z_cols_T, (1634793, 150), <class 'cupy.ndarray'>
ptl_Z_cols_T, (1634793, 150), <class 'cupy.ndarray'>
R_nz, (1634793,), <class 'cupy.ndarray'>
ptl_R_nz, (1634793,), <class 'cupy.ndarray'>


In [47]:
# W = X / (A Z) at nonzeros
W_data = vals / R_nz
W = cpx_sparse.coo_matrix((W_data, (rows, cols)), shape=X.shape)
ptl_W_data = vals / ptl_R_nz
ptl_W = cpx_sparse.coo_matrix((ptl_W_data, (rows, cols)), shape=X.shape)
inspect("W_data")
inspect("W")

W_data, (1634793,), <class 'cupy.ndarray'>
ptl_W_data, (1634793,), <class 'cupy.ndarray'>
W, (4000, 16000000), <class 'cupyx.scipy.sparse._coo.coo_matrix'>
ptl_W, (4000, 16000000), <class 'cupyx.scipy.sparse._coo.coo_matrix'>


In [48]:

# numerator = W @ Z^T   -> (I_mode, R)
numerator = W @ tl.transpose(Z)
inspect("W")
inspect("Z")
# ptl_numerator = ptl_W @ tl.transpose(ptl_Z)
ptl_numerator = ptl_W @ tl.transpose(cp.asarray(ptl_Z))
inspect("numerator")

# denominator = sum_j Z[r,j] broadcast to (I_mode, R)
den_row = tl.sum(Z, axis=1)  # (R,)
denominator = den_row[np.newaxis, :]
denominator = tl.clip(denominator, a_min=epsilon, a_max=None)
ptl_den_row = tl.sum(ptl_Z, axis=1)
ptl_denominator = ptl_den_row[np.newaxis, :]
ptl_denominator = tl.clip(ptl_denominator, a_min=epsilon, a_max=None)
inspect("den_row")
inspect("denominator")


W, (4000, 16000000), <class 'cupyx.scipy.sparse._coo.coo_matrix'>
ptl_W, (4000, 16000000), <class 'cupyx.scipy.sparse._coo.coo_matrix'>
Z, (150, 16000000), <class 'cupy.ndarray'>
ptl_Z, (150, 16000000), <class 'numpy.ndarray'>


OutOfMemoryError: Out of memory allocating 9,600,000,000 bytes (allocated so far: 20,255,523,840 bytes).

In [43]:
# Multiplicative update
new_A = A * (numerator / (denominator + 1e-12))
new_A = tl.clip(new_A, a_min=epsilon, a_max=None)
ptl_new_A = A * (ptl_numerator / (cp.asarray(ptl_denominator + 1e-12)))
ptl_new_A = tl.clip(ptl_new_A, a_min=epsilon, a_max=None)
inspect("new_A")

new_A, (2000, 150), <class 'cupy.ndarray'>
ptl_new_A, (2000, 150), <class 'cupy.ndarray'>


In [11]:
skip_factor_tensors = [ptl.tens2mat(ptl_tensor, mode) for mode in modes]
max_cuda_mem = torch.cuda.get_device_properties().total_memory
batch_rows = max_cuda_mem // (4000 ** 2 * 12)


In [5]:
ptl_factors = []
for mode in modes:
    # Sparse unfolding for this mode
    X = unfold_from_vectorized_sparse(tensor, shape, mode)
    core_np = tl.to_numpy(core)
    factors_np = [tl.to_numpy(f) for f in factors]
    # Dense reconstruction excluding current factor, unfolded along mode
    ptl_tucker = ptl.TuckerTensor(core=core_np,
                                  factors=factors_np)
    ptl_Z = ptl_tucker_to_tensor(ptl_tucker, skip_factor=mode)
    ptl_Z = ptl.tens2mat(ptl_Z, mode)
    # Z = tucker_to_tensor((core, factors), skip_factor=mode)
    # Z = unfold(Z, mode)  # (R, K) after unfold
    # inspect("Z")
    A = factors[mode]  # (I_mode, R)
    rows = X.row
    cols = X.col
    vals = X.data

    # Compute reconstruction only at nonzeros: R_nz = sum_r A[i,r] * Z[r,j]
    # A_rows: (nnz, R)
    A_rows = A[rows, :]
    # Z_cols_T: (nnz, R) because Z[:, cols] is (R, nnz)
    # Z_cols_T = tl.transpose(Z[:, cols])
    ptl_Z_cols_T = tl.transpose(ptl_Z[:, cols.get()])
    ptl_Z_cols_T = cp.asarray(ptl_Z_cols_T)
    # R_nz = tl.sum(A_rows * Z_cols_T, axis=1)
    # R_nz = tl.clip(R_nz, a_min=epsilon, a_max=None)
    ptl_R_nz = tl.sum(A_rows * ptl_Z_cols_T, axis=1)
    ptl_R_nz = tl.clip(ptl_R_nz, a_min=epsilon, a_max=None)
    # inspect("Z_cols_T")
    # inspect("R_nz")

    # W = X / (A Z) at nonzeros
    # W_data = vals / R_nz
    # W = cpx_sparse.coo_matrix((W_data, (rows, cols)), shape=X.shape)
    ptl_W_data = vals / ptl_R_nz
    ptl_W = cpx_sparse.coo_matrix((ptl_W_data, (rows, cols)), shape=X.shape)
    # inspect("W_data")
    # inspect("W")

    # numerator = W @ Z^T   -> (I_mode, R)
    # numerator = W @ tl.transpose(Z)
    # inspect("W")
    # inspect("Z")
    # ptl_numerator = ptl_W @ tl.transpose(ptl_Z)
    ptl_numerator = ptl_W @ tl.transpose(cp.asarray(ptl_Z))
    # inspect("numerator")

    # denominator = sum_j Z[r,j] broadcast to (I_mode, R)
    # den_row = tl.sum(Z, axis=1)  # (R,)
    # denominator = den_row[np.newaxis, :]
    # denominator = tl.clip(denominator, a_min=epsilon, a_max=None)
    ptl_den_row = tl.sum(ptl_Z, axis=1)
    ptl_denominator = ptl_den_row[np.newaxis, :]
    ptl_denominator = tl.clip(ptl_denominator, a_min=epsilon, a_max=None)
    # inspect("den_row")
    # inspect("denominator")

    # Multiplicative update
    # new_A = A * (numerator / (denominator + 1e-12))
    # new_A = tl.clip(new_A, a_min=epsilon, a_max=None)
    A = A * (ptl_numerator / (cp.asarray(ptl_denominator + 1e-12)))
    ptl_factors.append(A)

In [13]:
type(shape)

tuple

In [16]:
# if any of the elements in shape is over 1000, we print warning


In [6]:
from distance import kl_factor_update
reg_factors = []
for mode in modes:
    reg_factors.append(kl_factor_update(tensor, shape, core, factors, mode, epsilon))

In [10]:
for (ptl_factor, reg_factor) in zip(ptl_factors, reg_factors):
    print(type(ptl_factor), type(reg_factor))
    print(cp.allclose(ptl_factor, reg_factor))

<class 'cupy.ndarray'> <class 'cupy.ndarray'>
True
<class 'cupy.ndarray'> <class 'cupy.ndarray'>
True
<class 'cupy.ndarray'> <class 'cupy.ndarray'>
True
